In [0]:
%run ../transform/dim_fecha

In [0]:
%run ../transform/dim_canal

In [0]:
%run ../transform/dim_cliente

In [0]:
%run ../transform/dim_instrumento

In [0]:
TABLA_ORIGEN = "platform_dev.silver_byma.operaciones_replica"
tabla_destino = "platform_dev.gold_byma.fact_operaciones"

dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("fact_operaciones")

In [0]:
try:
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(f"La tabla {tabla_destino} no existe. Inicializando...")
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True
    elif carga_full:
        logger.warning(f"Carga full solicitada para la tabla de hechos {tabla_destino}.")
    else:
        logger.info(f"La tabla {tabla_destino} existe.")
        tiene_datos = spark.sql(f"SELECT 1 FROM {tabla_destino} LIMIT 1").take(1) != []
        if not tiene_datos:
            carga_full = True
except Exception:
    logger.exception(f"Error al validar infraestructura de hechos.")
    raise

In [0]:
logger.info("Iniciando extracción incremental y resolución de claves dimensionales...")

df_operaciones = spark.table(TABLA_ORIGEN)

if not carga_full:
    fila_control = spark.sql(f"SELECT ultima_fecha_procesada FROM platform_dev.bronce_byma.control_cargas WHERE proceso = '{tabla_destino}'").first()
    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
    else:
        ultima_fecha = "1970-01-01"
    
    logger.info(f"Procesando transacciones deltas posteriores a: {ultima_fecha}")
    df_operaciones_filtradas = df_operaciones.filter(F.col("fecha_particion") > F.lit(ultima_fecha))
else:
    df_operaciones_filtradas = df_operaciones

# 1. Preparar Cotizaciones usando ventana analítica para arrastrar el último precio disponible (Forward Fill)
df_cotizaciones_base = spark.table("platform_dev.silver_byma.cotizaciones_historicas")
window_ffill = Window.partitionBy("simbolo").orderBy("fecha").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_cotizaciones_robustas = df_cotizaciones_base \
        .withColumn("precio_cierre_limpio", F.last("precio_cierre", ignorenulls=True).over(window_ffill)) \
            .select(F.col("simbolo").alias("c_simbolo"), F.col("fecha").alias("c_fecha"), F.col("precio_cierre_limpio"))

# 2. Cargar Dimensiones para resolver las Claves Subrogadas (SK)
df_dim_inst = spark.table("platform_dev.gold_byma.dim_instrumento").select("instrumento_sk", "simbolo")
df_dim_clie = spark.table("platform_dev.gold_byma.dim_cliente").select("cliente_sk", "id_cliente")
df_dim_canal = spark.table("platform_dev.gold_byma.dim_canal").select("canal_sk", "nombre_canal")

# 3. Ensamble de Hechos mediante cruces controlados
df_enriquecido = (
    df_operaciones_filtradas
    # Cruce con Instrumentos
    .join(df_dim_inst, df_operaciones_filtradas["simbolo_titulo"] == df_dim_inst["simbolo"], "left")
    # Cruce con Clientes
    .join(df_dim_clie, "id_cliente", "left")
    # Cruce con Canales
    .join(df_dim_canal, df_operaciones_filtradas["origen"] == df_dim_canal["nombre_canal"], "left")
    # Cruce con Precios de Mercado Históricos (Data Real extraída en tu PC)
    .join(df_cotizaciones_robustas, (F.col("simbolo_titulo") == F.col("c_simbolo")) & (F.col("fecha_particion") == F.col("c_fecha")), "left")
    
    # Resolvemos clave temporal y cálculos requeridos por el negocio
    .withColumn("fecha_sk", F.date_format("fecha_particion", "yyyyMMdd").cast("int"))
    .withColumn("monto_total", F.col("cantidad") * F.col("precio"))
    .withColumn("precio_mercado", F.coalesce(F.col("precio_cierre_limpio"), F.col("precio")).cast("decimal(18,4)"))
    
    # Desvío Porcentual evitando fallas de división por cero
    .withColumn(
        "desvio_pct",
        F.expr("CAST(((precio - precio_mercado) / precio_mercado) * 100 AS DECIMAL(18,4))")
    )
    # Lógica Analítica de Banderas de Desvío (Compró Caro / Vendió Barato)
    .withColumn("flag_compro_caro",F.when((F.upper(F.col("tipoTran")) == "COMPRA") & (F.col("precio") > (F.col("precio_mercado") * F.lit(1.02))), True).otherwise(False))
    .withColumn("flag_vendio_barato",F.when((F.upper(F.col("tipoTran")) == "VENTA") & (F.col("precio") < (F.col("precio_mercado") * F.lit(0.98))), True).otherwise(False))
    
    # Renombrado final alineado estrictamente a los nombres de tu DDL física
    .select(
        F.col("id_transaccion"),
        F.col("instrumento_sk"),
        F.col("cliente_sk"),
        F.col("fecha_sk"),
        F.col("canal_sk"),
        F.col("cantidad"),
        F.col("precio").alias("precio_operado"),
        F.col("monto_total"),
        F.col("tipoTran").alias("tipo_operacion"),
        F.col("precio_mercado"),
        F.col("desvio_pct"),
        F.col("flag_compro_caro"),
        F.col("flag_vendio_barato"),
        F.current_timestamp().alias("_processed_at")
    )
)

df_enriquecido.createOrReplaceTempView("v_hechos_preparados")

In [0]:
cantidad_registros = df_enriquecido.count()

try:
    if carga_full:
        logger.warning(
            f"Iniciando reconstrucción COMPLETA y ATÓMICA de hechos en {tabla_destino} "
            f"para {cantidad_registros} registros."
        )
        
        # Un solo comando atómico: Cero downtime, máxima performance y protección ACID
        spark.sql(f"""
            INSERT OVERWRITE {tabla_destino}
            SELECT * FROM v_hechos_preparados
        """)
        
        logger.info("Carga full de hechos completada con éxito.")
    else:
        if cantidad_registros == 0:
            logger.info("No se detectaron transacciones nuevas para procesar en hechos.")
        else:
            logger.info(f"Carga incremental: Insertando/Actualizando {cantidad_registros} transacciones de mercado.")
            # El uso de MERGE por id_transaccion asegura que si el archivo vuelve a ingresar, no se dupliquen las métricas financieras
            spark.sql(f"""
                MERGE INTO {tabla_destino} AS target
                USING v_hechos_preparados AS source
                ON target.id_transaccion = source.id_transaccion
                WHEN MATCHED THEN UPDATE SET
                    target.precio_operado = source.precio_operado,
                    target.monto_total = source.monto_total,
                    target.precio_mercado = source.precio_mercado,
                    target.desvio_pct = source.desvio_pct,
                    target.flag_compro_caro = source.flag_compro_caro,
                    target.flag_vendio_barato = source.flag_vendio_barato,
                    target._processed_at = source._processed_at
                WHEN NOT MATCHED THEN
                  INSERT *
            """)
            logger.info("Impacto incremental de hechos finalizado.")
except Exception:
    logger.exception(f"Error crítico en fase de persistencia de hechos.")
    raise

In [0]:
ultima_fecha_origen = df_operaciones.agg(F.max("fecha_particion").alias("max_fecha")).first()["max_fecha"]

if ultima_fecha_origen is not None and cantidad_registros > 0:
    spark.sql(f"""
        MERGE INTO platform_dev.bronce_byma.control_cargas AS target
        USING (SELECT '{tabla_destino}' AS proceso, DATE('{ultima_fecha_origen}') AS u_fecha) AS source
        ON target.proceso = source.proceso
        WHEN MATCHED THEN UPDATE SET target.ultima_fecha_procesada = source.u_fecha, target.fecha_actualizacion = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (proceso, ultima_fecha_procesada, fecha_actualizacion) VALUES (source.proceso, source.u_fecha, current_timestamp())
    """)
    logger.info(f"Frontera temporal de hechos actualizada a la fecha: {ultima_fecha_origen}")